In [ ]:
import langchain, langchain_community
print(langchain.__version__)
print(langchain_community.__version__)

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [ ]:
import utils

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
# A real time tool would call real mapping API (Google Maps etc) to get live congestion and ETA 
@tool
def TrafficStatusTool(route: str) -> str:
    """Check current traffic conditions for a delivery route."""
    return f"Traffic on {route} is currently moderate. Approx delay: 10 minutes."

# A real time tool would pull live telematics/WMS/ERP data (FedEx-style truck load data)  
@tool
def VehicleCapacityTool(vehicle_id: str) -> str:
    """Check remaining load capacity of a specified delivery vehicle."""
    return f"Vehicle {vehicle_id} has 120 package capacity remaining."

# A real time tool would query CRM/order system for actual customer SLA + priority rules
@tool
def DeliveryWindowTool(address: str) -> str:
    """Check allowed customer delivery window for a specific address."""
    return f"Delivery window for {address}: 2PM–5PM."


tools = [TrafficStatusTool, VehicleCapacityTool, DeliveryWindowTool]

In [ ]:
DEL_SYSTEM_PROMPT = """
You are a logistics optimization AI assistant.
Your goal: plan the MOST efficient delivery route considering:
- Traffic patterns
- Delivery windows
- Vehicle capacity
- Time efficiency
Use a deliberate reasoning format:
Thought:
Action: <tool-name>
Action Input: <arguments>

Repeat until ready.
When fully confident, respond using this format:

Final Answer:
<optimized delivery plan>
"""

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", DEL_SYSTEM_PROMPT),
    ("human",  "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

In [ ]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)

# 'verbose' shows step logs, and 'max_iterations' limits how many reasoning steps it can take.
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [ ]:
result = executor.invoke({
    "input": "Plan route for vehicle V-20 to deliver to 10 Sarojini Nagar and 15 Lajpat Nagar"
})

print("\nOPTIMIZED PLAN")
print(result)